# Federated Learning on 20 Newsgroups
## Per-Class Shapley-based Free-Rider Detection

---

## 0. Environment Setup

In [ ]:
!git clone https://github.com/isaac-sun/20NEWS-FL.git 2>/dev/null; true
%cd /content/20NEWS-FL
!git pull origin main

In [ ]:
!pip install -q openpyxl tqdm seaborn

In [ ]:
import torch, sklearn
print(f'PyTorch: {torch.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Using: {DEVICE}')

---
## 1. Data Loading

In [ ]:
import numpy as np
from utils.seed import set_seed
from data.newsgroups import load_newsgroups

set_seed(42)
train_ds, val_ds, test_ds, input_dim, train_labels = load_newsgroups(max_features=10000, val_ratio=0.1)
print(f'Input dim: {input_dim}, Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')
print(f'Classes: {len(np.unique(train_labels))}')

---
## 2. Model & Partition

In [ ]:
from models.mlp import MLP
from utils.partition import iid_partition
from torch.utils.data import Subset

NUM_CLIENTS = 10
model = MLP(input_dim, 256, 20).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

set_seed(42)
partition = iid_partition(train_ds, NUM_CLIENTS)
for cid in range(NUM_CLIENTS):
    print(f'  Client {cid}: {len(partition[cid])} samples')

---
## 3. Attack Definitions (Paper-Aligned)

| Attack | Paper | Formula |
|--------|-------|---------|
| DFR | Fraboni [11] | g = sigma * t^{-gamma} * N(0,I) |
| SDFR | Zhu [12] | U_f = (norm(delta_t)/norm(delta_prev)) * delta_t |
| AFR | Zhu [12] | SDFR + sparse noise on d coords, magnitude from E[cos beta] |

---
## 4. Run All Experiments

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import copy, matplotlib.pyplot as plt, pandas as pd, seaborn as sns
from collections import deque
from tqdm import tqdm
from config import Config
from utils.seed import set_seed
from data.newsgroups import load_newsgroups
from models.mlp import MLP
from fl.client import FLClient
from fl.server import FLServer
from fl.aggregation import fedavg_aggregate
from attacks.dfr import dfr_attack, estimate_dfr_sigma
from attacks.sdfr import sdfr_attack
from attacks.afr import afr_attack, AFRState
from contribution.shapley import (
    estimate_round_shapley_per_class, per_class_to_overall,
    compute_class_metrics, _class_weights_from_loader
)
from detection.utility_score import UtilityScoreTracker
from utils.partition import iid_partition, non_iid_partition
from utils.export import export_results
from torch.utils.data import DataLoader, Subset

import numpy as np
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


In [ ]:
def apply_attack(attack_type, global_sd, global_history, cfg,
                 round_num=1, dfr_sigma_est=None,
                 afr_state=None, val_loss_t=None):
    """Generate free-rider update. Returns (update, metadata_dict)."""
    if attack_type == "dfr":
        sigma = cfg.dfr_sigma
        if cfg.dfr_estimate_sigma and dfr_sigma_est is not None:
            sigma = dfr_sigma_est
        update = dfr_attack(global_sd, sigma=sigma,
                            round_num=round_num, gamma=cfg.dfr_gamma)
        return update, {}
    elif attack_type == "sdfr":
        update = sdfr_attack(global_sd, global_history)
        return update, {}
    elif attack_type == "afr":
        e_cos_beta = 0.0
        if cfg.afr_e_cos_beta_override is not None:
            e_cos_beta = cfg.afr_e_cos_beta_override
        elif afr_state is not None and val_loss_t is not None:
            e_cos_beta = afr_state.get_e_cos_beta(val_loss_t)
        mean_base_norm = None
        if afr_state is not None:
            mean_base_norm = afr_state.get_mean_base_norm()
        update, base_norm = afr_attack(
            global_sd, global_history,
            n_total=cfg.num_clients,
            e_cos_beta=e_cos_beta,
            mean_base_norm=mean_base_norm,
            noisy_frac=cfg.afr_noisy_frac,
        )
        return update, {"afr_base_norm": base_norm}
    raise ValueError(f"Unknown: {attack_type}")


def run_experiment(config, train_dataset, val_dataset, test_dataset,
                   input_dim, train_labels):
    """Returns: round_details, summary, test_accs, test_losses, per_class_records"""
    set_seed(config.seed)
    print(f"\n{'='*60}")
    print(f"Experiment: {config.experiment_name} | Attack: {config.attack_type}")
    print(f"{'='*60}")

    if config.iid:
        partition = iid_partition(train_dataset, config.num_clients)
    else:
        partition = non_iid_partition(train_labels, config.num_clients)

    client_datasets = {cid: Subset(train_dataset, idx) for cid, idx in partition.items()}
    model_fn = lambda: MLP(input_dim, config.hidden_dim, config.num_classes).to(config.device)
    model = model_fn()
    server = FLServer(model, val_dataset, test_dataset, config)
    clients = {cid: FLClient(cid, client_datasets[cid], model_fn, config)
               for cid in range(config.num_clients)}

    num_mal = int(config.num_clients * config.malicious_ratio) if config.attack_type != "none" else 0
    malicious_ids = set(range(num_mal))

    utility_tracker = UtilityScoreTracker(alpha=config.utility_alpha)
    round_details = []
    per_class_records = []
    test_accs, test_losses = [], []
    participation_counts = {cid: 0 for cid in range(config.num_clients)}
    cumulative_sv = {cid: 0.0 for cid in range(config.num_clients)}
    cumulative_class_var = {cid: [] for cid in range(config.num_clients)}
    cumulative_pos_sum = {cid: [] for cid in range(config.num_clients)}

    # Attack state trackers
    dfr_sigma_est = None
    afr_state = AFRState(ema_alpha=config.afr_base_norm_ema_alpha) if config.attack_type == "afr" else None

    val_loader = DataLoader(val_dataset, batch_size=config.batch_size)
    eval_model = model_fn()
    class_weights = _class_weights_from_loader(val_loader, config.num_classes)

    for round_t in tqdm(range(config.num_rounds), desc=config.experiment_name):
        selected = server.select_clients(config.num_clients, config.participation_ratio)
        global_sd = server.get_global_state_dict()
        global_history = list(server.global_history)

        # DFR sigma auto-estimation (once, from initial global delta)
        if config.attack_type == "dfr" and dfr_sigma_est is None and config.dfr_estimate_sigma:
            est = estimate_dfr_sigma(global_sd, global_history)
            if est is not None:
                dfr_sigma_est = est
                print(f"  DFR sigma auto-estimated: {dfr_sigma_est:.6f}")

        # AFR: pre-attack validation loss for E[cos β] estimation
        val_loss_t = None
        if config.attack_type == "afr":
            val_loss_t, _ = server.evaluate_val()
            if afr_state is not None and afr_state.val_loss_init is None:
                afr_state.val_loss_init = val_loss_t
                print(f"  AFR val_loss_init: {val_loss_t:.4f}")

        updates = {}
        afr_base_norms = []
        for cid in selected:
            participation_counts[cid] += 1
            if cid in malicious_ids:
                update, meta = apply_attack(
                    config.attack_type, global_sd, global_history, config,
                    round_num=round_t + 1,
                    dfr_sigma_est=dfr_sigma_est,
                    afr_state=afr_state,
                    val_loss_t=val_loss_t,
                )
                updates[cid] = update
                if "afr_base_norm" in meta:
                    afr_base_norms.append(meta["afr_base_norm"])
            else:
                updates[cid] = clients[cid].train(global_sd)

        # Post-attack: update AFR state
        if afr_state is not None and val_loss_t is not None and afr_base_norms:
            afr_state.update(val_loss_t, float(np.mean(afr_base_norms)))

        per_class_sv = estimate_round_shapley_per_class(
            eval_model, updates, global_sd, val_loader,
            num_classes=config.num_classes, server_lr=config.server_lr,
            num_mc_samples=config.num_mc_samples, device=config.device,
        )
        shapley_vals = per_class_to_overall(per_class_sv, class_weights)
        class_metrics = compute_class_metrics(per_class_sv)
        utility_tracker.update(shapley_vals)

        new_sd = fedavg_aggregate(global_sd, updates, config.server_lr)
        server.update_global_model(new_sd)

        test_loss, test_acc = server.evaluate()
        test_accs.append(test_acc)
        test_losses.append(test_loss)

        if round_t % 5 == 0 or round_t == config.num_rounds - 1:
            print(f"  Round {round_t:>2d}: acc={test_acc:.4f}  loss={test_loss:.4f}")

        for cid in selected:
            sv = shapley_vals.get(cid, 0.0)
            cumulative_sv[cid] += sv
            pc = participation_counts[cid]
            cm = class_metrics.get(cid, {})
            cls_var = cm.get("class_sv_variance", 0.0)
            pos_sum = cm.get("positive_class_sv_sum", 0.0)
            cumulative_class_var[cid].append(cls_var)
            cumulative_pos_sum[cid].append(pos_sum)

            round_details.append({
                "experiment_name": config.experiment_name,
                "attack_type": config.attack_type,
                "round": round_t,
                "client_id": cid,
                "is_malicious": cid in malicious_ids,
                "participation_count_so_far": pc,
                "round_shapley_value": sv,
                "cumulative_shapley_value": cumulative_sv[cid],
                "mean_shapley_value": cumulative_sv[cid] / pc,
                "class_sv_variance": cls_var,
                "positive_class_sv_sum": pos_sum,
                "mean_class_sv_variance": float(np.mean(cumulative_class_var[cid])),
                "mean_positive_class_sv_sum": float(np.mean(cumulative_pos_sum[cid])),
                "utility_score": utility_tracker.scores.get(cid, 0.0),
            })

            pc_arr = per_class_sv.get(cid, np.zeros(config.num_classes))
            rec = {
                "experiment_name": config.experiment_name,
                "round": round_t,
                "client_id": cid,
                "is_malicious": cid in malicious_ids,
            }
            for c in range(config.num_classes):
                rec[f"class_{c}"] = float(pc_arr[c])
            per_class_records.append(rec)

    honest_ids = [c for c in range(config.num_clients) if c not in malicious_ids]
    def _avg_mean_sv(ids):
        vals = [cumulative_sv[c] / max(participation_counts[c], 1) for c in ids]
        return float(np.mean(vals)) if vals else 0.0
    def _avg_cum_sv(ids):
        return float(np.mean([cumulative_sv[c] for c in ids])) if ids else 0.0
    def _avg_metric(ids, store):
        vals = [float(np.mean(store[c])) if store[c] else 0.0 for c in ids]
        return float(np.mean(vals)) if vals else 0.0

    summary = {
        "experiment_name": config.experiment_name,
        "attack_type": config.attack_type,
        "malicious_ratio": config.malicious_ratio if num_mal > 0 else 0.0,
        "final_global_accuracy": test_accs[-1],
        "final_global_loss": test_losses[-1],
        "avg_round_shapley_honest": _avg_mean_sv(honest_ids),
        "avg_round_shapley_malicious": _avg_mean_sv(list(malicious_ids)),
        "avg_cumulative_shapley_honest": _avg_cum_sv(honest_ids),
        "avg_cumulative_shapley_malicious": _avg_cum_sv(list(malicious_ids)),
        "shapley_gap": _avg_mean_sv(honest_ids) - _avg_mean_sv(list(malicious_ids)),
        "avg_class_sv_variance_honest": _avg_metric(honest_ids, cumulative_class_var),
        "avg_class_sv_variance_malicious": _avg_metric(list(malicious_ids), cumulative_class_var),
        "avg_positive_class_sv_sum_honest": _avg_metric(honest_ids, cumulative_pos_sum),
        "avg_positive_class_sv_sum_malicious": _avg_metric(list(malicious_ids), cumulative_pos_sum),
        "attack_effective": "",
        "notes": "",
    }
    return round_details, summary, test_accs, test_losses, per_class_records


In [ ]:
base_cfg = Config(
    num_clients=10, num_rounds=30, local_epochs=3, local_lr=0.001,
    server_lr=1.0, participation_ratio=0.8, batch_size=64,
    hidden_dim=256, num_classes=20, max_features=10000, val_ratio=0.1,
    malicious_ratio=0.4, num_mc_samples=30, seed=42,
    device=DEVICE, results_dir="results",
)

experiments = [
    ("baseline_no_attack", "none"),
    ("attack_dfr", "dfr"),
    ("attack_sdfr", "sdfr"),
    ("attack_afr", "afr"),
]

all_details, all_summaries, all_curves, all_pc = [], [], {}, []

set_seed(42)
train_ds, val_ds, test_ds, input_dim, train_labels = load_newsgroups(
    max_features=base_cfg.max_features, val_ratio=base_cfg.val_ratio
)

for exp_name, attack_type in experiments:
    cfg = copy.deepcopy(base_cfg)
    cfg.experiment_name = exp_name
    cfg.attack_type = attack_type
    details, summary, accs, losses, pc_records = run_experiment(
        cfg, train_ds, val_ds, test_ds, input_dim, train_labels
    )
    all_details.extend(details)
    all_summaries.append(summary)
    all_curves[exp_name] = {"acc": accs, "loss": losses}
    all_pc.extend(pc_records)

baseline_acc = all_summaries[0]["final_global_accuracy"]
for s in all_summaries[1:]:
    drop = baseline_acc - s["final_global_accuracy"]
    s["attack_effective"] = drop > 0.02
    s["notes"] = f"accuracy drop: {drop:.4f}"

df = pd.DataFrame(all_details)
df_pc = pd.DataFrame(all_pc)
class_cols = [f"class_{c}" for c in range(20)]
attack_names = ["attack_dfr", "attack_sdfr", "attack_afr"]

print("\n" + "=" * 72)
print("All experiments done!")
print("=" * 72)

---
## 5. Summary Table

In [ ]:
df_summary = pd.DataFrame(all_summaries)
df_summary[['experiment_name','final_global_accuracy','final_global_loss',
    'shapley_gap','avg_class_sv_variance_honest','avg_class_sv_variance_malicious',
    'avg_positive_class_sv_sum_honest','avg_positive_class_sv_sum_malicious']]

---
## 6. Accuracy & Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for name, data in all_curves.items():
    ax1.plot(data["acc"], label=name, linewidth=2)
ax1.set_xlabel("Round"); ax1.set_ylabel("Test Accuracy")
ax1.set_title("Global Test Accuracy"); ax1.legend(); ax1.grid(True, alpha=0.3)
for name, data in all_curves.items():
    ax2.plot(data["loss"], label=name, linewidth=2)
ax2.set_xlabel("Round"); ax2.set_ylabel("Test Loss")
ax2.set_title("Global Test Loss"); ax2.legend(); ax2.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

---
## 7. Round-Level Shapley: Honest vs Malicious

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    honest = sub[~sub["is_malicious"]].groupby("round")["round_shapley_value"].mean()
    mal = sub[sub["is_malicious"]].groupby("round")["round_shapley_value"].mean()
    label = exp_name.replace("attack_", "").upper()
    axes[0].plot(honest.index, honest.values, label=f"{label} honest", linewidth=2)
    axes[0].plot(mal.index, mal.values, ls="--", label=f"{label} malicious", linewidth=2)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].set_xlabel("Round"); axes[0].set_ylabel("Mean Round SV")
axes[0].set_title("Round-Level Shapley Value"); axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

bar_data = []
for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    last = sub[sub["round"] == sub["round"].max()]
    label = exp_name.replace("attack_", "").upper()
    h = last[~last["is_malicious"]]["cumulative_shapley_value"].mean()
    m = last[last["is_malicious"]]["cumulative_shapley_value"].mean()
    bar_data.append((f"{label}\nhonest", h, "steelblue"))
    bar_data.append((f"{label}\nmalicious", m, "coral"))
x = np.arange(len(bar_data))
axes[1].bar(x, [b[1] for b in bar_data], color=[b[2] for b in bar_data], alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels([b[0] for b in bar_data], fontsize=9)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].set_ylabel("Cumulative SV"); axes[1].set_title("Cumulative Shapley Value")
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout(); plt.show()

---
## 8. Shapley Heatmap (Per-Client Per-Round)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
axes = axes.flatten()
for idx, (exp_name, _) in enumerate(experiments):
    ax = axes[idx]
    sub = df[df["experiment_name"] == exp_name]
    if sub.empty: ax.set_title(exp_name); continue
    pivot = sub.pivot_table(index="client_id", columns="round",
                            values="round_shapley_value", aggfunc="mean")
    mal_ids = set(sub[sub["is_malicious"]]["client_id"].unique())
    labels = [f"{cid}{chr(9733) if cid in mal_ids else ''}" for cid in pivot.index]
    sns.heatmap(pivot, ax=ax, cmap="RdBu_r", center=0,
                xticklabels=5, yticklabels=labels, cbar_kws={"shrink": 0.8})
    ax.set_xlabel("Round"); ax.set_ylabel("Client ID")
    ax.set_title(f"{exp_name}\n(star = malicious)")
    for i, cid in enumerate(pivot.index):
        if cid in mal_ids:
            ax.add_patch(plt.Rectangle((-0.5, i-0.5), pivot.shape[1], 1,
                fill=False, edgecolor="red", linewidth=1.5))
fig.suptitle("Per-Client Per-Round Shapley Value Heatmap", fontsize=15, y=1.02)
fig.tight_layout(); plt.show()

---
## 9. Per-Class SV Fingerprint

Honest clients contribute unevenly across classes (peaked profile); free-riders have near-zero flat profiles.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
_raw = fetch_20newsgroups(subset="train")
class_names = [n.split(".")[-1][:8] for n in _raw.target_names]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for idx, exp_name in enumerate(attack_names):
    ax = axes[idx]
    sub_pc = df_pc[df_pc["experiment_name"] == exp_name]
    honest_pc = sub_pc[~sub_pc["is_malicious"]][class_cols].mean().values
    mal_pc = sub_pc[sub_pc["is_malicious"]][class_cols].mean().values
    x = np.arange(20)
    w = 0.35
    ax.bar(x - w/2, honest_pc, w, label="Honest", color="steelblue", alpha=0.8)
    ax.bar(x + w/2, mal_pc, w, label="Malicious", color="coral", alpha=0.8)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=75, fontsize=6)
    ax.set_ylabel("Mean Per-Class SV")
    label = exp_name.replace("attack_", "").upper()
    ax.set_title(f"{label}: Per-Class SV Fingerprint")
    ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
fig.suptitle("Per-Class SV Fingerprint: Honest (peaked) vs Free-Rider (flat)", fontsize=13)
fig.tight_layout(); plt.show()

---
## 10. Two-Metric Scatter Plot

X = Class SV Variance, Y = Positive Class SV Sum. Each point = one client (last 5 rounds avg).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, exp_name in enumerate(attack_names):
    ax = axes[idx]
    sub = df[df["experiment_name"] == exp_name]
    last_rounds = sub[sub["round"] >= sub["round"].max() - 4]
    client_stats = last_rounds.groupby(["client_id", "is_malicious"]).agg({
        "class_sv_variance": "mean",
        "positive_class_sv_sum": "mean",
    }).reset_index()
    honest = client_stats[~client_stats["is_malicious"]]
    mal = client_stats[client_stats["is_malicious"]]
    ax.scatter(honest["class_sv_variance"], honest["positive_class_sv_sum"],
               c="steelblue", s=120, label="Honest", edgecolors="navy", zorder=5)
    ax.scatter(mal["class_sv_variance"], mal["positive_class_sv_sum"],
               c="coral", s=120, marker="X", label="Malicious", edgecolors="darkred", zorder=5)
    ax.set_xlabel("Class SV Variance"); ax.set_ylabel("Positive Class SV Sum")
    label = exp_name.replace("attack_", "").upper()
    ax.set_title(f"{label}"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle("Two-Metric Scatter: Honest (circle) vs Malicious (X)", fontsize=13)
fig.tight_layout(); plt.show()

---
## 11. Box Plots: Metric Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, exp_name in enumerate(attack_names):
    sub = df[df["experiment_name"] == exp_name]
    label = exp_name.replace("attack_", "").upper()
    ax = axes[0, idx]
    data_h = sub[~sub["is_malicious"]]["class_sv_variance"]
    data_m = sub[sub["is_malicious"]]["class_sv_variance"]
    bp = ax.boxplot([data_h, data_m], labels=["Honest", "Malicious"], patch_artist=True, widths=0.6)
    bp["boxes"][0].set_facecolor("steelblue"); bp["boxes"][1].set_facecolor("coral")
    for b in bp["boxes"]: b.set_alpha(0.7)
    ax.set_ylabel("Class SV Variance"); ax.set_title(f"{label}: Variance")
    ax.grid(True, axis="y", alpha=0.3)
    ax = axes[1, idx]
    data_h = sub[~sub["is_malicious"]]["positive_class_sv_sum"]
    data_m = sub[sub["is_malicious"]]["positive_class_sv_sum"]
    bp = ax.boxplot([data_h, data_m], labels=["Honest", "Malicious"], patch_artist=True, widths=0.6)
    bp["boxes"][0].set_facecolor("steelblue"); bp["boxes"][1].set_facecolor("coral")
    for b in bp["boxes"]: b.set_alpha(0.7)
    ax.set_ylabel("Positive Class SV Sum"); ax.set_title(f"{label}: Positive Sum")
    ax.grid(True, axis="y", alpha=0.3)
fig.suptitle("Metric Distributions: Honest vs Malicious", fontsize=13)
fig.tight_layout(); plt.show()

---
## 12. Per-Class SV Heatmap (Single Client Deep Dive)

DFR experiment: honest vs malicious client across 20 classes x 30 rounds.

In [ ]:
sub_pc = df_pc[df_pc["experiment_name"] == "attack_dfr"]
honest_cids = sorted(sub_pc[~sub_pc["is_malicious"]]["client_id"].unique())
mal_cids = sorted(sub_pc[sub_pc["is_malicious"]]["client_id"].unique())
h_cid, m_cid = honest_cids[0], mal_cids[0]

from sklearn.datasets import fetch_20newsgroups
_raw = fetch_20newsgroups(subset="train")
class_names = [n.split(".")[-1][:8] for n in _raw.target_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))
for ax, cid, title in [(ax1, h_cid, f"Honest (Client {h_cid})"),
                        (ax2, m_cid, f"Malicious (Client {m_cid})")]:
    cdata = sub_pc[sub_pc["client_id"] == cid]
    pivot = cdata.pivot_table(index="round", values=class_cols)
    pivot_t = pivot.T
    pivot_t.index = class_names
    vmax = max(abs(pivot_t.values.min()), abs(pivot_t.values.max()), 1e-6)
    sns.heatmap(pivot_t, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                xticklabels=5, cbar_kws={"shrink": 0.8})
    ax.set_xlabel("Round"); ax.set_ylabel("Class")
    ax.set_title(f"DFR: {title}")
fig.suptitle("Per-Class Per-Round SV: Honest (varied) vs Free-Rider (blank)", fontsize=13)
fig.tight_layout(); plt.show()

---
## 13. Core Metrics Over Rounds

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    h = sub[~sub["is_malicious"]].groupby("round")["class_sv_variance"].mean()
    m = sub[sub["is_malicious"]].groupby("round")["class_sv_variance"].mean()
    label = exp_name.replace("attack_", "").upper()
    axes[0,0].plot(h.index, h.values, label=f"{label} honest", linewidth=2)
    axes[0,0].plot(m.index, m.values, ls="--", label=f"{label} malicious", linewidth=2)
axes[0,0].set_xlabel("Round"); axes[0,0].set_ylabel("Class SV Variance")
axes[0,0].set_title("Per-Class SV Variance"); axes[0,0].legend(fontsize=7); axes[0,0].grid(True, alpha=0.3)

for exp_name in attack_names:
    sub = df[df["experiment_name"] == exp_name]
    h = sub[~sub["is_malicious"]].groupby("round")["positive_class_sv_sum"].mean()
    m = sub[sub["is_malicious"]].groupby("round")["positive_class_sv_sum"].mean()
    label = exp_name.replace("attack_", "").upper()
    axes[0,1].plot(h.index, h.values, label=f"{label} honest", linewidth=2)
    axes[0,1].plot(m.index, m.values, ls="--", label=f"{label} malicious", linewidth=2)
axes[0,1].set_xlabel("Round"); axes[0,1].set_ylabel("Positive Class SV Sum")
axes[0,1].set_title("Positive Per-Class SV Sum"); axes[0,1].legend(fontsize=7); axes[0,1].grid(True, alpha=0.3)

for metric, ax_idx, ylabel in [
    ("mean_class_sv_variance", (1,0), "Mean Class SV Variance"),
    ("mean_positive_class_sv_sum", (1,1), "Mean Positive Class SV Sum"),
]:
    bl, bv, bc = [], [], []
    for exp_name in attack_names:
        sub = df[df["experiment_name"] == exp_name]
        last = sub[sub["round"] == sub["round"].max()]
        label = exp_name.replace("attack_", "").upper()
        hv = last[~last["is_malicious"]][metric].mean()
        mv = last[last["is_malicious"]][metric].mean()
        bl += [f"{label}\nhonest", f"{label}\nmalicious"]
        bv += [hv, mv]; bc += ["steelblue", "coral"]
    x = np.arange(len(bl))
    axes[ax_idx].bar(x, bv, color=bc, alpha=0.85)
    axes[ax_idx].set_xticks(x); axes[ax_idx].set_xticklabels(bl, fontsize=8)
    axes[ax_idx].set_ylabel(ylabel); axes[ax_idx].set_title(ylabel)
    axes[ax_idx].grid(True, axis="y", alpha=0.3)
fig.tight_layout(); plt.show()

---
## 14. Per-Client Cumulative SV Bar Chart

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for idx, (exp_name, _) in enumerate(experiments):
    sub = df[df["experiment_name"] == exp_name]
    if sub.empty: axes[idx].set_title(exp_name); continue
    last = sub[sub["round"] == sub["round"].max()].sort_values("client_id")
    colors = ["coral" if r["is_malicious"] else "steelblue" for _, r in last.iterrows()]
    axes[idx].bar(last["client_id"], last["cumulative_shapley_value"], color=colors, alpha=0.85)
    axes[idx].set_xlabel("Client ID"); axes[idx].set_ylabel("Cumulative SV")
    axes[idx].set_title(exp_name)
    axes[idx].axhline(y=0, color='k', linewidth=0.5); axes[idx].grid(True, axis="y", alpha=0.3)
fig.suptitle("Per-Client Cumulative SV (blue=honest, red=malicious)", fontsize=14)
fig.tight_layout(); plt.show()

---
## 15. Export Excel Report

In [ ]:
os.makedirs("results", exist_ok=True)
filepath = export_results(all_details, all_summaries, "results")
print(f"Excel saved: {filepath}")
try:
    from google.colab import files
    files.download(filepath)
    print("Download triggered")
except ImportError:
    print("(Not Colab, skip auto-download)")

---
## Final Summary

In [ ]:
print("=" * 110)
print("EXPERIMENT SUMMARY")
print("=" * 110)
print(f"{'Experiment':<22s}  {'Acc':>6s}  {'Loss':>6s}  {'SV_h':>10s}  {'SV_m':>10s}  "
      f"{'Var_h':>10s}  {'Var_m':>10s}  {'PosSum_h':>10s}  {'PosSum_m':>10s}")
print("-" * 110)
for s in all_summaries:
    print(
        f"  {s['experiment_name']:<22s}"
        f"  {s['final_global_accuracy']:.4f}"
        f"  {s['final_global_loss']:.4f}"
        f"  {s['avg_round_shapley_honest']:+.6f}"
        f"  {s['avg_round_shapley_malicious']:+.6f}"
        f"  {s.get('avg_class_sv_variance_honest',0):.2e}"
        f"  {s.get('avg_class_sv_variance_malicious',0):.2e}"
        f"  {s.get('avg_positive_class_sv_sum_honest',0):.6f}"
        f"  {s.get('avg_positive_class_sv_sum_malicious',0):.6f}"
    )
print("=" * 110)
print("\nKey metrics:")
print("  Var: Per-class SV variance (honest > malicious = good)")
print("  PosSum: Positive per-class SV sum (honest > malicious = good)")

---
## Custom Experiments

| Param | Description | Default |
|-------|-------------|------|
| `num_clients` | Number of clients | 10 |
| `num_rounds` | Communication rounds | 30 |
| `local_epochs` | Local training epochs | 3 |
| `local_lr` | Local learning rate | 0.001 |
| `participation_ratio` | Per-round participation | 0.8 |
| `malicious_ratio` | Malicious ratio | **0.4** |
| `dfr_sigma` | DFR initial noise sigma | **0.5** |
| `dfr_gamma` | DFR decay exponent | **1.0** |
| `afr_e_cos_beta` | AFR expected cosine sim | **0.5** |
| `afr_noisy_frac` | AFR fraction of perturbed params | **0.1** |
| `num_mc_samples` | MC Shapley samples | 30 |
| `iid` | IID partition | True |